# Sensitivity tests: forest cover, slope, perennial snow, and the robust twins

How the per-range results depend on the choices the analyses make, using the class keys of the sparse table
(partials schema 3) as views, so every test is a re-selection of the same pixels:

1. **Forest-cover threshold.** The analyses keep pixels with FCF ≤ 50 %. The same statistics with FCF ≤ 10, 20 and 30 %.
2. **Slope class.** Aspect control per slope class (it should vanish on flat ground), and the quality of the
   Sentinel-1 record on steep slopes: the share of terrain with a mapped onset, the MAD and the per-year coverage.
3. **Seasonal versus perennial snow** (ESA WorldCover class 70): each regime's statistics separately.
4. **Robust twins** of two results quoted with Pearson r and OLS: the lapse–aspect co-variation (Spearman, with its
   shuffle null) and the early-melt sharpening of aspect control (Theil–Sen per range).

```
reads                                                       writes (results/<version>/)
─────                                                       ──────────────────────────
data/aggregation/<v>/sparse_mountain_ranges.parquet ──┐      sensitivity_forest_threshold_by_range.csv, _summary.csv
results/<v>/mountain_range_statistics.csv ────────────┤      sensitivity_slope_class_by_range.csv, _summary.csv
results/<v>/mountain_range_band_aspect.csv ───────────┴──►   sensitivity_perennial_by_range.csv, _summary.csv
                                                             sensitivity_robust_twins.json
```

Every summary is over the included ranges (the inclusion rule of `range_statistics.ipynb`): counts and medians of
per-range values, never pooled fits. Run `range_statistics.ipynb` first.

In [ ]:
import json

import numpy as np
import pandas as pd
import scipy.stats as sps

from gsro_analysis import aggregate, paths, range_metrics as rm, results, settings

In [ ]:
config = settings.load_config()
VERSION = config.version
WATER_YEARS = [int(y) for y in config.water_years]
UNIT = 'mountain_ranges'
aggregation_dir = paths.aggregation_dir(UNIT, VERSION)
results_dir = paths.resultsdir(UNIT, VERSION)
sparse_path = aggregation_dir / f'sparse_{UNIT}.parquet'

FCF_THRESHOLDS = [10, 20, 30, 50]              # % forest cover; FCF classes are 10 % wide, so FCF <= T is classes 0 .. T/10 - 1
OPEN = aggregate.FILTER_VIEWS['fcf_lte_50']
N_PERM = 2000
SEED = 0
MIN_PERENNIAL_KM2 = 100                        # a range enters the perennial comparison with this much mapped perennial snow

In [ ]:
statistics_df = pd.read_csv(results_dir / 'mountain_range_statistics.csv').set_index('name')
id_to_name = dict(zip(statistics_df['GMBA_V2_ID'].astype(int), statistics_df.index))
included = statistics_df.index[statistics_df['included'].astype(bool)]
temperature_df = statistics_df[[f'spring_temperature_anomaly_WY{y}' for y in WATER_YEARS]].set_axis(WATER_YEARS, axis=1)
print(f'{len(included)} included ranges')


def named(ds):
    # the view's unit axis (GMBA ids) as range names; ranges outside the statistics table dropped
    names = [id_to_name.get(int(i)) for i in ds['mountain_range'].values]
    return ds.assign_coords(mountain_range=names).sel(mountain_range=[n is not None for n in names])


def per_range_summary(view_ds):
    # the per-range quantities of the main table, recomputed on a view
    snow_km2 = view_ds['runoff_onset_median_n'].fillna(0).sum(rm.BIN_DIMS).to_pandas() * rm.PIXEL_AREA_KM2
    onset = aggregate.weighted_mean(view_ds, 'runoff_onset_median', rm.BIN_DIMS).to_pandas()
    mad = aggregate.weighted_mean(view_ds, 'runoff_onset_mad', rm.BIN_DIMS).to_pandas()
    lapse = rm.lapse_rate_weighted_bins(view_ds)
    sunny = rm.sunny_shady(view_ds)[0]['sunny_shady_d']
    anomaly = rm.range_year_anomaly(view_ds)
    sens = rm.temperature_sensitivity(anomaly, temperature_df.reindex(anomaly.index), n_perm=1)
    return pd.DataFrame({'snow_km2': snow_km2, 'median_onset_dowy': onset, 'mad_d': mad, 'lapse_rate_per_100m': lapse,
                         'sunny_shady_d': sunny, 'ols_slope': sens['ols_slope'], 'ols_p': sens['ols_p'],
                         'pearson_r': sens['pearson_r'], 'theil_sen_slope': sens['theil_sen_slope'], 'sensitivity_n': sens['sensitivity_n']})


def summarise(df):
    f = df.dropna(subset=['ols_slope'])
    return {'ranges': int(len(df)), 'with_fit': int(len(f)), 'median_snow_km2': float(df['snow_km2'].median()),
            'median_onset_dowy': float(df['median_onset_dowy'].median()), 'median_mad_d': float(df['mad_d'].median()),
            'median_lapse_rate': float(df['lapse_rate_per_100m'].median()), 'median_sunny_shady_d': float(df['sunny_shady_d'].median()),
            'pct_negative': float(100 * (f['ols_slope'] < 0).mean()), 'sig_negative': int(((f['ols_p'] < 0.05) & (f['ols_slope'] < 0)).sum()),
            'sig_positive': int(((f['ols_p'] < 0.05) & (f['ols_slope'] > 0)).sum()), 'median_slope': float(f['ols_slope'].median()),
            'median_r': float(f['pearson_r'].median())}

## 1. Forest-cover threshold

The same included ranges, each statistic recomputed with only the pixels at or below each forest-cover threshold.

In [ ]:
keys = ('elevation', 'aspect')
by_threshold = {}
for threshold in FCF_THRESHOLDS:
    classes = list(range(threshold // 10))
    view_ds = rm.base_view(named(aggregate.view(aggregate.load_sparse(sparse_path, keys, fcf=classes), UNIT, keys, WATER_YEARS, fcf=classes)))
    by_threshold[threshold] = per_range_summary(view_ds)
forest_threshold_df = pd.concat(by_threshold, names=['fcf_max_pct', 'name']).reset_index()
forest_threshold_summary_df = pd.DataFrame({t: summarise(d.reindex(included)) for t, d in by_threshold.items()}).T
forest_threshold_summary_df.index.name = 'fcf_max_pct'
# how much each range's sensitivity moves relative to the analyses' 50 % threshold
ref = by_threshold[50]['ols_slope']
forest_threshold_summary_df['median_abs_slope_change_vs_50'] = [float((by_threshold[t]['ols_slope'] - ref).reindex(included).abs().median()) for t in FCF_THRESHOLDS]
forest_threshold_df.assign(**results.provenance(config)).to_csv(results_dir / 'sensitivity_forest_threshold_by_range.csv', index=False, float_format='%.4g')
forest_threshold_summary_df.assign(**results.provenance(config)).to_csv(results_dir / 'sensitivity_forest_threshold_summary.csv', float_format='%.4g')
forest_threshold_summary_df.round(2)

## 2. Slope class: aspect control and the quality of the record on steep terrain

Per range and slope class (FCF ≤ 50 %): the share of the class's terrain with a mapped onset, the sunny–shady
difference (2A, elevation removed), the pixel-weighted MAD, the mean per-year coverage, and the share of the class's
pixels in bins that pass the 100-pixel rule (splitting by slope makes bins smaller).

In [ ]:
slope_ds = named(aggregate.view(aggregate.load_sparse(sparse_path, ('elevation', 'aspect', 'slope_class'), fcf=OPEN), UNIT,
                                ('elevation', 'aspect'), WATER_YEARS, fcf=OPEN, keep=('slope_class',)))
rows = []
for label in aggregate.SLOPE_CLASSES:
    raw = slope_ds.sel(slope_class=label)
    v = rm.base_view(raw)
    n = raw['runoff_onset_median_n'].fillna(0)
    terrain = raw['terrain_n'].fillna(0)
    passing = n.where(n >= rm.MIN_PIXELS_PER_BIN, 0)
    coverage = rm.coverage(v)
    df = pd.DataFrame({'slope_class': label,
                       'snow_km2': n.sum(rm.BIN_DIMS).to_pandas() * rm.PIXEL_AREA_KM2,
                       'mapped_share_of_terrain': (n.sum(rm.BIN_DIMS) / terrain.sum(rm.BIN_DIMS).where(terrain.sum(rm.BIN_DIMS) > 0)).to_pandas(),
                       'share_in_passing_bins': (passing.sum(rm.BIN_DIMS) / n.sum(rm.BIN_DIMS).where(n.sum(rm.BIN_DIMS) > 0)).to_pandas(),
                       'sunny_shady_d': rm.sunny_shady(v)[0]['sunny_shady_d'],
                       'mad_d': aggregate.weighted_mean(v, 'runoff_onset_mad', rm.BIN_DIMS).to_pandas(),
                       'mean_coverage': coverage.mean(axis=1)})
    rows.append(df)
slope_class_df = pd.concat(rows).rename_axis('name').reset_index()
slope_class_df['snow_share'] = slope_class_df['snow_km2'] / slope_class_df.groupby('name')['snow_km2'].transform('sum')
inc = slope_class_df[slope_class_df['name'].isin(included)]
slope_class_summary_df = inc.groupby('slope_class', sort=False).agg(
    ranges=('sunny_shady_d', lambda s: int(s.notna().sum())), median_snow_share=('snow_share', 'median'),
    median_mapped_share_of_terrain=('mapped_share_of_terrain', 'median'), median_share_in_passing_bins=('share_in_passing_bins', 'median'),
    median_sunny_shady_d=('sunny_shady_d', 'median'), median_mad_d=('mad_d', 'median'), median_mean_coverage=('mean_coverage', 'median'))
slope_class_df.assign(**results.provenance(config)).to_csv(results_dir / 'sensitivity_slope_class_by_range.csv', index=False, float_format='%.4g')
slope_class_summary_df.assign(**results.provenance(config)).to_csv(results_dir / 'sensitivity_slope_class_summary.csv', float_format='%.4g')
slope_class_summary_df.round(3)

In [ ]:
# within each range, does the sunny-shady difference grow with slope? (Spearman across the classes with a value)
rho = {}
for name, g in inc.dropna(subset=['sunny_shady_d']).groupby('name'):
    if len(g) >= 4:
        rho[name] = sps.spearmanr(range(len(g)), g['sunny_shady_d']).statistic
rho = pd.Series(rho)
slope_trend = {'ranges': int(len(rho)), 'median_spearman_rho': float(rho.median()), 'share_positive': float((rho > 0).mean())}
slope_trend

## 3. Seasonal versus perennial snow

Pixels on ESA WorldCover permanent snow and ice (class 70) are kept in every analysis; here each regime gets its own
statistics, for the included ranges with at least `MIN_PERENNIAL_KM2` km² of mapped perennial snow.

In [ ]:
regimes = {}
for code_, label in enumerate(aggregate.PERENNIAL_CLASSES):
    v = rm.base_view(named(aggregate.view(aggregate.load_sparse(sparse_path, keys, fcf=OPEN, perennial=[code_]), UNIT, keys, WATER_YEARS,
                                          fcf=OPEN, perennial=[code_])))
    regimes[label] = per_range_summary(v)
perennial_df = pd.concat(regimes, names=['regime', 'name']).reset_index()
wide = perennial_df.pivot(index='name', columns='regime')
both = wide.index[(wide[('snow_km2', 'perennial')] >= MIN_PERENNIAL_KM2) & wide.index.isin(included)]
perennial_summary_df = pd.DataFrame({label: summarise(regimes[label].reindex(both)) for label in regimes}).T
perennial_summary_df.index.name = 'regime'
perennial_summary_df['ranges_compared'] = len(both)
perennial_summary_df.loc['perennial', 'median_onset_shift_vs_seasonal_d'] = float((wide.loc[both, ('median_onset_dowy', 'perennial')] - wide.loc[both, ('median_onset_dowy', 'seasonal')]).median())
perennial_summary_df.loc['perennial', 'perennial_share_of_snow_median'] = float((wide.loc[both, ('snow_km2', 'perennial')] / wide.loc[both, 'snow_km2'].sum(axis=1)).median())
perennial_df.assign(**results.provenance(config)).to_csv(results_dir / 'sensitivity_perennial_by_range.csv', index=False, float_format='%.4g')
perennial_summary_df.assign(**results.provenance(config)).to_csv(results_dir / 'sensitivity_perennial_summary.csv', float_format='%.4g')
perennial_summary_df.round(2)

## 4. Robust twins

**Lapse–aspect co-variation with Spearman ρ.** The per-range Pearson r of `range_statistics.ipynb`, recomputed as a
rank correlation across the years, with the same within-range shuffle null. **Early-melt sharpening with Theil–Sen.**
The per-range slopes of `aspect_seasonality.ipynb`, recomputed as Theil–Sen slopes.

In [ ]:
yearly_df = pd.read_csv(results_dir / 'mountain_range_yearly_topography.csv')
yearly_df = yearly_df[yearly_df['water_year'].astype(str) != 'median'].astype({'water_year': int})
lapse_y = yearly_df.pivot(index='name', columns='water_year', values='lapse_rate_per_100m')
amp_y = yearly_df.pivot(index='name', columns='water_year', values='sunny_shady_d')
rng = np.random.default_rng(SEED)
pairs = []
for r in included:
    if r in lapse_y.index and r in amp_y.index:
        ok = lapse_y.loc[r].notna() & amp_y.loc[r].notna()
        if ok.sum() >= rm.MIN_YEARS_INTERANNUAL:
            pairs.append((r, lapse_y.loc[r][ok].values, amp_y.loc[r][ok].values))
obs = np.array([sps.spearmanr(x, y).statistic for _, x, y in pairs])
null_share = [np.mean([sps.spearmanr(x, rng.permutation(y)).statistic > 0 for _, x, y in pairs]) for _ in range(N_PERM // 4)]
covariation_spearman = {'ranges': len(pairs), 'median_rho': float(np.median(obs)), 'share_positive': float((obs > 0).mean()),
                        'null_share_positive_p95': float(np.percentile(null_share, 95)), 'n_perm': N_PERM // 4}
covariation_spearman

In [ ]:
band_df = pd.read_csv(results_dir / 'mountain_range_band_aspect.csv')
band_df['water_year'] = band_df['water_year'].astype(str)
med = band_df[band_df['water_year'] == 'median'][['name', 'elevation', 'onset_dowy', 'amp_d']].rename(columns={'onset_dowy': 'om', 'amp_d': 'am'})
yr = band_df[band_df['water_year'] != 'median'].merge(med, on=['name', 'elevation']).dropna(subset=['onset_dowy', 'amp_d', 'om', 'am'])
yr = yr[yr['name'].isin(included)]
yr['d_date'] = yr['onset_dowy'] - yr['om']; yr['d2a'] = 2 * (yr['amp_d'] - yr['am']); yr['from_equinox'] = yr['om'] - 172
classes = [(-90, -30, '> 30 d before the equinox'), (-30, 30, 'within 30 d of the equinox'), (30, 60, '30-60 d after'),
           (60, 90, '60-90 d after'), (90, 200, '> 90 d after')]
early_melt_ts = {}
for lo, hi, label in classes:
    g = yr[(yr['from_equinox'] >= lo) & (yr['from_equinox'] < hi)]
    sl = np.array([-10 * sps.theilslopes(r['d2a'], r['d_date'])[0] for _, r in g.groupby('name') if len(r) >= 20 and r['d_date'].std() > 0])
    early_melt_ts[label] = {'ranges': int(len(sl)), 'median_d_per_10d_earlier': float(np.median(sl)) if len(sl) else None,
                            'share_positive': float((sl > 0).mean()) if len(sl) else None}
early_melt_ts

In [ ]:
twins = {'lapse_aspect_covariation_spearman': covariation_spearman, 'early_melt_theil_sen': early_melt_ts,
         'aspect_vs_slope_class_within_range': slope_trend, '_provenance': results.provenance(config)}
with open(results_dir / 'sensitivity_robust_twins.json', 'w') as f:
    json.dump(twins, f, indent=1)
print(json.dumps({k: v for k, v in twins.items() if k != '_provenance'}, indent=1))